In [2]:
# Import required packages
import mne
import numpy as np
import matplotlib.pyplot as plt
from mne.time_frequency import tfr_morlet
import os
from pathlib import Path
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("MNE version:", mne.__version__)

ModuleNotFoundError: No module named 'mne'

In [ ]:
# Configuration
SUBJECT_ID = '02'
CONDITIONS = ['joyful', 'neutral']  # Adjust based on actual event names
BIDS_ROOT = Path('ds005107')  # Change this path if needed

# Frequency bands of interest based on literature
FREQ_BANDS = {
    'theta': (4, 8),
    'alpha': (8, 13), 
    'beta': (13, 30)
}

In [ ]:
def load_subject_data(subject_id):
    """
    Load subject data from BIDS format
    """
    print(f"Loading data for subject {subject_id}...")
    
    subject_path = BIDS_ROOT / f'sub-{subject_id}'
    if not subject_path.exists():
        raise FileNotFoundError(f"Subject path not found: {subject_path}")
    
    # Find all sessions for this subject
    sessions = [d.name for d in subject_path.iterdir() if d.is_dir() and d.name.startswith('ses-')]
    print(f"Found sessions: {sessions}")
    
    all_epochs = []
    for session in sessions:
        session_path = subject_path / session / 'meg'
        raw_files = list(session_path.glob('*.fif'))
        
        for raw_file in raw_files:
            print(f"  Loading {raw_file.name}")
            try:
                # Load raw data
                raw = mne.io.read_raw_fif(raw_file, preload=True, verbose=False)
                
                # Get events
                events, event_id = mne.events_from_annotations(raw, verbose=False)
                print(f"    Events found: {event_id}")
                
                # Check if we have the conditions we need
                valid_conditions = [cond for cond in CONDITIONS if cond in event_id]
                if valid_conditions:
                    # Create epochs
                    epochs = mne.Epochs(raw, events, event_id, 
                                      tmin=-0.2, tmax=1.0,
                                      baseline=(-0.2, 0),
                                      preload=True,
                                      verbose=False)
                    all_epochs.append(epochs)
                    
            except Exception as e:
                print(f"    Error loading {raw_file}: {e}")
                continue
    
    if not all_epochs:
        raise ValueError("No valid epochs found for the specified conditions")
    
    # Combine epochs from all sessions
    epochs_combined = mne.concatenate_epochs(all_epochs)
    print(f"Combined epochs: {len(epochs_combined)} trials")
    return epochs_combined


In [ ]:
# Load the data
epochs = load_subject_data(SUBJECT_ID)

# Display basic information
print("\nEpochs info:")
print(f"Number of trials: {len(epochs)}")
print(f"Time points per trial: {len(epochs.times)}")
print(f"Sampling rate: {epochs.info['sfreq']} Hz")
print(f"Channels: {len(epochs.ch_names)}")
print(f"Events: {epochs.event_id}")

In [ ]:
# Plot sensor locations
fig = epochs.plot_sensors(show_names=True, show=True)
plt.title(f'Sensor Locations - Subject {SUBJECT_ID}')

In [ ]:
# Plot some example epochs to check data quality
fig = epochs.plot(n_epochs=5, n_channels=10, scalings=dict(mag=1e-12))
plt.suptitle('Example Epochs - Raw Data Quality Check')

In [ ]:
def time_frequency_analysis(epochs, conditions):
    """
    Perform time-frequency analysis on specified conditions
    """
    # Define frequencies of interest
    freqs = np.logspace(*np.log10([4, 30]), num=20)
    n_cycles = freqs / 3.
    
    tfr_results = {}
    
    for condition in conditions:
        if condition not in epochs.event_id:
            print(f"Condition {condition} not found. Available: {list(epochs.event_id.keys())}")
            continue
            
        print(f"Processing condition: {condition} ({len(epochs[condition])} trials)")
        
        # Time-frequency decomposition
        power = tfr_morlet(epochs[condition], freqs=freqs, n_cycles=n_cycles,
                          use_fft=True, return_itc=False, decim=3, 
                          n_jobs=1, verbose=False)
        
        # Baseline correction
        power.apply_baseline(baseline=(-0.2, 0), mode='percent')
        
        tfr_results[condition] = power
    
    return tfr_results

In [ ]:
# Perform time-frequency analysis
print("Starting time-frequency analysis...")
tfr_results = time_frequency_analysis(epochs, CONDITIONS)

In [ ]:
def plot_condition_power(tfr_results, condition):
    """
    Plot time-frequency power for a single condition
    """
    if condition not in tfr_results:
        return
    
    power = tfr_results[condition]
    
    # Plot average power across all sensors
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'Time-Frequency Power - {condition.capitalize()} Condition', fontsize=16)
    
    # Average across all magnetometer sensors
    picks = mne.pick_types(power.info, meg='mag')
    avg_power = power.average(picks=picks)
    
    # Time-frequency plot
    ax = axes[0, 0]
    avg_power.plot([0], baseline=None, axes=ax, show=False, colorbar=True)
    ax.set_title(f'Average Power (all mag sensors)')
    
    # Topomaps at specific time points
    times = [0.1, 0.3, 0.5]  # 100ms, 300ms, 500ms
    for i, time in enumerate(times):
        ax = axes[0, 1] if i == 0 else axes[1, i-1]
        time_idx = np.argmin(np.abs(power.times - time))
        freq_idx = (power.freqs >= 4) & (power.freqs <= 30)
        
        # Average across frequency band for topography
        topo_data = power.data[:, freq_idx, time_idx].mean(axis=1)
        
        im = mne.viz.plot_topomap(topo_data, power.info, axes=ax, show=False)
        ax.set_title(f'{time*1000:.0f} ms')
        plt.colorbar(im[0], ax=ax)
    
    plt.tight_layout()
    return fig

In [ ]:
# Plot individual conditions
for condition in CONDITIONS:
    if condition in tfr_results:
        plot_condition_power(tfr_results, condition)


In [ ]:
def plot_comparison(tfr_results):
    """
    Plot comparison between conditions
    """
    if len(tfr_results) < 2:
        print("Need at least two conditions for comparison")
        return
    
    conditions = list(tfr_results.keys())
    
    # Create difference (joyful - neutral)
    if 'joyful' in tfr_results and 'neutral' in tfr_results:
        diff_power = tfr_results['joyful'].copy()
        diff_power._data = tfr_results['joyful'].data - tfr_results['neutral'].data
        
        # Plot comparison
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle('Time-Frequency Analysis: Joyful vs Neutral Faces', fontsize=16)
        
        # Plot each condition and difference
        for i, (cond, power) in enumerate([('joyful', tfr_results['joyful']), 
                                         ('neutral', tfr_results['neutral']),
                                         ('difference', diff_power)]):
            
            # Average across magnetometers
            picks = mne.pick_types(power.info, meg='mag')
            avg_power = power.average(picks=picks)
            
            # Time-frequency plot
            row = i // 2
            col = i % 2 + (0 if i < 2 else 1)
            ax = axes[row, col]
            
            avg_power.plot([0], baseline=None, axes=ax, show=False, colorbar=True)
            ax.set_title(f'{cond.capitalize()} Condition')
        
        plt.tight_layout()
        return fig

In [ ]:
# Plot comparison between conditions
comparison_fig = plot_comparison(tfr_results)


In [ ]:
def statistical_analysis(tfr_results, condition1='joyful', condition2='neutral'):
    """
    Perform statistical comparison between conditions
    """
    if condition1 not in tfr_results or condition2 not in tfr_results:
        print(f"Cannot compare {condition1} vs {condition2}")
        return None
    
    data1 = tfr_results[condition1].data  # (epochs, channels, freqs, times)
    data2 = tfr_results[condition2].data
    
    # T-test across epochs
    t_stats, p_values = stats.ttest_ind(data1, data2, axis=0)
    
    print(f"Statistical Comparison: {condition1} vs {condition2}")
    print("=" * 60)
    
    # Analyze each frequency band
    freqs = tfr_results[condition1].freqs
    times = tfr_results[condition1].times
    
    results_summary = {}
    
    for band_name, (f_low, f_high) in FREQ_BANDS.items():
        band_mask = (freqs >= f_low) & (freqs <= f_high)
        
        # Key time windows based on literature
        time_windows = {
            'early': (0.15, 0.25),    # 150-250ms - emotional processing
            'mid': (0.25, 0.45),      # 250-450ms - sustained processing  
            'late': (0.45, 0.8)       # 450-800ms - later integration
        }
        
        print(f"\n{band_name.upper()} BAND ({f_low}-{f_high} Hz):")
        print("-" * 40)
        
        band_results = {}
        
        for window_name, (t_start, t_end) in time_windows.items():
            time_mask = (times >= t_start) & (times <= t_end)
            
            # Count significant channels (p < 0.05, uncorrected)
            window_p_values = p_values[:, band_mask, :][:, :, time_mask]
            sig_channels = np.any(window_p_values < 0.05, axis=(1, 2))
            n_sig = np.sum(sig_channels)
            
            # Effect size (mean difference)
            mean_diff = np.mean(data1[:, band_mask, :][:, :, time_mask] - 
                              data2[:, band_mask, :][:, :, time_mask])
            
            print(f"  {window_name} ({t_start*1000:.0f}-{t_end*1000:.0f}ms):")
            print(f"    Significant channels: {n_sig}/{len(sig_channels)}")
            print(f"    Mean difference: {mean_diff:.4f}")
            print(f"    Min p-value: {np.min(window_p_values):.4f}")
            
            band_results[window_name] = {
                'n_sig_channels': n_sig,
                'mean_diff': mean_diff,
                'min_p_value': np.min(window_p_values)
            }
        
        results_summary[band_name] = band_results
    
    return p_values, results_summary


In [ ]:
# Perform statistical analysis
print("Performing statistical analysis...")
p_values, results_summary = statistical_analysis(tfr_results)


In [ ]:
def plot_significance_maps(tfr_results, p_values, condition1='joyful', condition2='neutral'):
    """
    Plot significance maps for the comparison
    """
    if condition1 not in tfr_results or condition2 not in tfr_results:
        return
    
    freqs = tfr_results[condition1].freqs
    times = tfr_results[condition1].times
    
    # Create figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Statistical Significance: {condition1.capitalize()} vs {condition2.capitalize()}', fontsize=16)
    
    # Plot for each frequency band
    for i, (band_name, (f_low, f_high)) in enumerate(FREQ_BANDS.items()):
        band_mask = (freqs >= f_low) & (freqs <= f_high)
        
        # Average p-values across channels for visualization
        band_p_values = p_values[:, band_mask, :].mean(axis=0)
        
        # Plot significance map
        ax = axes[i // 2, i % 2]
        im = ax.imshow(-np.log10(band_p_values + 1e-10),  # Avoid log(0)
                      extent=[times[0], times[-1], f_high, f_low],
                      aspect='auto', cmap='Reds')
        
        ax.set_title(f'{band_name.capitalize()} Band ({f_low}-{f_high} Hz)')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Frequency (Hz)')
        plt.colorbar(im, ax=ax, label='-log10(p-value)')
        
        # Add significance threshold line
        ax.axhline(y=(f_low + f_high)/2, color='blue', linestyle='--', alpha=0.5)
        ax.axvline(x=0.2, color='blue', linestyle='--', alpha=0.5, label='Stimulus')
    
    plt.tight_layout()
    return fig


In [ ]:
# Plot significance maps
sig_fig = plot_significance_maps(tfr_results, p_values)


In [ ]:
# Summary of key findings based on literature expectations
print("\n" + "="*70)
print("SUMMARY OF KEY FINDINGS")
print("="*70)

print("\nBased on literature review expectations:")
print("✓ Theta (4-8 Hz): Should show enhancement for emotional faces around 150-250ms")
print("✓ Alpha (8-13 Hz): May show attention-related modulations")  
print("✓ Beta (13-30 Hz): Possible differences in later time windows")

print("\nOur results:")
for band_name, band_results in results_summary.items():
    print(f"\n{band_name.upper()} BAND:")
    for window_name, results in band_results.items():
        sig_star = " ***" if results['min_p_value'] < 0.05 else ""
        print(f"  {window_name}: {results['n_sig_channels']} sig channels, "
              f"p_min = {results['min_p_value']:.4f}{sig_star}")

In [ ]:
# Save results for future reference
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)

# Save figures
if 'comparison_fig' in locals():
    comparison_fig.savefig(output_dir / 'time_frequency_comparison.png', dpi=300, bbox_inches='tight')

if 'sig_fig' in locals():
    sig_fig.savefig(output_dir / 'statistical_significance.png', dpi=300, bbox_inches='tight')

# Save summary statistics
import json
with open(output_dir / 'analysis_summary.json', 'w') as f:
    summary = {
        'subject': SUBJECT_ID,
        'conditions_analyzed': list(tfr_results.keys()),
        'total_trials': len(epochs),
        'results_summary': results_summary
    }
    json.dump(summary, f, indent=2)

print(f"\nResults saved to '{output_dir}/' directory")

In [ ]:
# Final sanity check: Basic data quality metrics
print("\n" + "="*70)
print("DATA QUALITY METRICS")
print("="*70)

print(f"Subject: {SUBJECT_ID}")
print(f"Total trials: {len(epochs)}")
print(f"Sampling rate: {epochs.info['sfreq']} Hz")
print(f"Data duration: {len(epochs) * 1.2:.1f} seconds")  # 1.2s per trial
print(f"Number of sensors: {len(epochs.ch_names)}")

# Check for each condition
for condition in CONDITIONS:
    if condition in epochs.event_id:
        n_trials = len(epochs[condition])
        print(f"Trials in {condition} condition: {n_trials}")
